In [ ]:
import os
import yaml
import polars as pl
import pandas as pd
from tqdm import tqdm
from anngeno import AnnGeno

## ProteinGym DMS indel scores

In [ ]:
dms_dir = '/s/project/deeprvat/ukb_gym/DMS_scores_indels_zeroshot/'

pg_list = []
for filename in tqdm(os.listdir(dms_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pd.read_csv(f'{dms_dir}{filename}')
            temp['protein_name'] = filename.split('_')[0]
            # temp = temp.with_columns(pl.lit(filename.split('_')[0]).alias('gene_name'))
            pg_list.append(temp)

pgdf = pd.concat(pg_list)
pgdf

## ProteinGym DMS SNP scores

In [ ]:
dms_dir = '/s/project/deeprvat/ukb_gym/DMS_scores_SNPs_zeroshot/'

pg_list = []
for filename in tqdm(os.listdir(dms_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pd.read_csv(f'{dms_dir}{filename}')
            temp['protein_name'] = filename.split('_')[0]
            # temp = temp.with_columns(pl.lit(filename.split('_')[0]).alias('gene_name'))
            pg_list.append(temp)

pgdf = pd.concat(pg_list)
pgdf[['protein_name', 'mutant', 'DMS_score']]

In [ ]:
pgdf[pgdf['mutant']=='C229R']

In [ ]:
gene_df = pd.read_parquet('/s/project/deeprvat/deeprvat_input/protein_coding_genes.parquet').rename(columns={"id": "region"})
gene_df['gene_id'] = gene_df['gene'].str.split('.').str[0]
gene_df = gene_df.drop(columns=['gene'])
gene_df

In [ ]:
dms_df = pgdf[['protein_name', 'mutant', 'DMS_score']].merge(gene_df, left_on='protein_name', right_on='gene_name')
print(dms_df['gene_name'].nunique())
dms_df

In [ ]:
p2g = pd.read_csv('protein2gene_ids.tsv', sep='\t')[['protein_name', 'gene_name']]

missed = pgdf[~pgdf['protein_name'].isin(dms_df['protein_name'])][['protein_name', 'mutant', 'DMS_score']].merge(p2g, on='protein_name').merge(gene_df, left_on='gene_name', right_on='gene_name')

missed

In [ ]:
all_dms = pd.concat([dms_df, missed])
all_dms

## Read anngeno to get relevant variants

In [ ]:
config_path = '../config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

anngeno_file = config.get("anngeno_file")
ag = AnnGeno(filename=anngeno_file, filemode="r")

# Subset to genes with protein_gym scores
dms_genes = all_dms['region'].unique()
# maf = 0.001
variants_to_keep = set(ag.annotations.query("(region in @dms_genes)")["id"])
ag.subset_variants(variants_to_keep)

ag.annotations

## get amino acid substitutions from VEP

In [ ]:
# Write as VEP input file

chunk_vcf_file = '/s/project/deeprvat/ukb_gym/DMS_scores_SNPs_zeroshot/dms_genes.vcf'

with open(chunk_vcf_file, "w") as f:
    # Write the VCF header
    f.write("##fileformat=VCFv4.0\n")
    f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")

    # Write each variant without ID, QUAL, FILTER, or INFO
    for _, row in ag.annotations.iterrows():
        vcf_row = f'{row["chrom"]}\t{row["pos"]}\t{row["id"]}\t{row["ref"]}\t{row["alt"]}\t.\t.\t.\n'
        f.write(vcf_row)

print("All chunks processed.")

## Read VEP results

In [ ]:
# vep = pd.read_csv('/s/project/deeprvat/ukb_gym/DMS_scores_SNPs_zeroshot/dms_genes_annotated.vcf', sep='\t')
anno = pl.read_csv('/s/project/deeprvat/ukb_gym/DMS_scores_SNPs_zeroshot/dms_genes_annotated.vcf', separator='\t', comment_prefix='##', infer_schema_length=500000).rename({'#Uploaded_variation':'id', 'Gene':'gene_id'}).to_pandas()
anno = anno[anno['Amino_acids'].str.contains('/')]
anno

In [ ]:
anno['mutant_position'] = anno['Protein_position'].str.split('/').str[0]
anno['mutant_position']

In [ ]:
anno['mutant'] = anno.apply(lambda row: row['Amino_acids'].replace('/', str(row['mutant_position'])), axis=1)
anno['mutant']

In [ ]:
ukb_pg = all_dms.merge(anno[['mutant', 'id', 'Location', 'Allele', 'gene_id']], on=['gene_id', 'mutant'])
ukb_pg

In [ ]:
ukb_pg[ukb_pg['id']==10807299]

In [ ]:
temp = ag.annotations[['id', 'chrom', 'pos', 'ref', 'alt', 'region']].merge(ukb_pg[['id', 'DMS_score']], on='id')
temp

In [ ]:
temp['DMS_score'].hist(bins=50, log=True)

In [ ]:
# Choose DMS score with highest absolute value

temp['abs_DMS_score'] = temp['DMS_score'].abs()
result = temp.loc[temp.groupby('id')['abs_DMS_score'].idxmax()]
result

In [ ]:
result[result['region'].isin([8495, 15986])].shape

In [ ]:
result[['id', 'chrom', 'pos', 'ref', 'alt', 'region', 'DMS_score']].to_parquet('/s/project/deeprvat/ukb_gym/new_annotations/proteinGym_DMS_scores.parquet')

In [ ]:
result

## Intersect with genebass

In [ ]:
gb_res = pd.read_parquet('/s/project/deeprvat/ukb_gym/genebass/genebass_all_associations_p_e-5.parquet')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

# Filter genebass results to get just the significant ones that we want
gb_sig = gb_res.query("(significant == True) & (modifier != 'custom')")
# gb_sig = gb_res.query("(significant == True) & (trait_type=='continuous') & (modifier != 'custom')")
gb_sig = gb_sig[gb_sig['gene_id'].isin(ukb_pg['gene_id'].unique())]
gb_sig

In [ ]:
gb_sig[['gene_id', 'description', 'phenocode', 'modifier']].drop_duplicates()

In [ ]:
config_path = '../config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

phenotypes = config.get("phenotypes_for_testing")
phenotypes

In [ ]:
gb_sig['phenotype'] = gb_sig['description'].str.replace(' ', '_').str.replace('-', '_').str.replace('(','').str.replace(')','')
pheno_df = gb_sig[(gb_sig['phenotype'].isin(phenotypes))]
pheno_df

In [ ]:
pheno_df[['gene_id', 'phenotype', 'phenocode', 'modifier']].drop_duplicates()

In [ ]:
gene_df = pd.read_parquet('/s/project/deeprvat/deeprvat_input/protein_coding_genes.parquet').rename(columns={"id": "region"})
gene_df['gene_id'] = gene_df['gene'].str.split('.').str[0]
gene_df = gene_df.drop(columns=['gene'])
gene_df


In [ ]:
pheno_df.merge(gene_df, on='gene_id')[['region', 'gene_id', 'gene_name', 'phenotype', 'phenocode']].drop_duplicates().to_parquet('/s/project/deeprvat/ukb_gym/proteinGym_benchmark_21assocs.parquet')